In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from combra import data
from combra.metrics import (
    metrics_vs_n,
    convergence_stats, print_convergence_report,
    summarize_metric_distribution,
    plot_wdist_convergence_grid, plot_metric_distribution,
)
from combra.metrics.compare import find_ethalon

# Angle binning step (degrees). Used by generation AND the W-dist computation.
# generate_angles overwrites parquets, so changing STEP and re-running generation
# drops any previously-stored step. Files already containing STEP are skipped.
STEP = 2.0
ANGLES_ROOT = Path('./data/angles')

def sweep_dir(h5_path):
    return ANGLES_ROOT / (Path(h5_path).stem + '_msl5')


In [ ]:
# Per-resolution sources for the 3x3 convergence grid. Each row = one resolution;
# the 'real' H5 doubles as ethalon (largest-N parquet) and as source of the
# original self-convergence curve. Diffit h5s match 4_grid_plot.ipynb's SOURCES
# so wdist values agree with that notebook's table.
SOURCES = {
    256: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
        'san':    './data/h5/gen_san_256x256_N100_000.h5',
        'diffit': './data/h5/00017-diffit-256-gpus2-batch192_N10000.h5',
    },
    512: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
        'san':    './data/h5/gen_san_512x512_N100_000.h5',
        'diffit': './data/h5/00018-diffit-512-gpus4-batch256_N10000.h5',
    },
    1024: {
        'real':   './data/h5/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360.h5',
    },
}

# N sweep per (kind, resolution). Bounded by per-class dataset size of each H5 —
# diffit at 512 only has 1k/class so its sweep stops there. All kinds start at
# N=100 so curves share the same x-axis start. The largest N in each list also
# serves as the ethalon for that (resolution, kind).
N_VALUES = {
    'real':   {256:  [100, 150, 200, 250, 300, 360],
               512:  [100, 150, 200, 250, 300, 360],
               1024: [100, 150, 200, 250, 300, 360]},
    'san':    {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
    'diffit': {256:  [100, 250, 360, 500, 1000, 2500, 5000, 10000],
               512:  [100, 250, 360, 500, 1000, 2500, 5000, 10000]},
}

# Diffit and san were trained on the same 3 classes but indexed them in
# different orders, so each generator needs its own class_map. Verified
# against 4_grid_plot.ipynb's GEN_NAME_FOR_PER_MODE.
CLASS_MAP_PER_KIND = {
    'san':    {'class_0': 'class_Ultra_Co25',
               'class_1': 'class_Ultra_Co11',
               'class_2': 'class_Ultra_Co6_2'},
    'diffit': {'class_0': 'class_Ultra_Co11',
               'class_1': 'class_Ultra_Co25',
               'class_2': 'class_Ultra_Co6_2'},
}

types_dict = {
    'Ultra_Co11':  'small grain',
    'Ultra_Co25':  'medium grain',
    'Ultra_Co8':   'medium-small grain',
    'Ultra_Co6_2': 'large grain',
    'Ultra_Co15':  'medium-small grain',
}

# Plot config — columns of the 3x3 grid (the 3 grain classes diffit was trained on).
CLASSES = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']
GRAIN_LABEL = {'class_Ultra_Co11':  'small grain',
               'class_Ultra_Co25':  'medium grain',
               'class_Ultra_Co6_2': 'large grain'}
LABELS = {'real': 'original', 'san': 'san', 'diffit': 'diffit '}

# Stats config (used in the convergence + gain-distribution sections).
METRICS = ['w_dist', 'mu1', 'mu2', 'sigma1', 'sigma2', 'amp1', 'amp2']
KINDS = ['san', 'diffit']
# Main "plateau-regime" endpoints used for rel_err_abs_% and the T3 main column.
ENDPOINTS_BY_KIND = {'real': (100, 300), 'san': (1000, 10000), 'diffit': (1000, 10000)}
# Pre-plateau endpoints — earlier-N pair shown alongside the main one in T3.
# Captures the steep convergence regime before the curves flatten.
PRE_ENDPOINTS_BY_KIND = {'san': (360, 1000), 'diffit': (360, 1000)}
KIND_DISPLAY = {'real': 'Real', 'san': 'SAN', 'diffit': 'DiffiT'}

# Generation

For every (resolution, source) in `SOURCES`, extract angle distributions at every N in `N_VALUES[kind]`. Output: `./data/angles/<h5-stem>_msl5/angles_n{N}.parquet`.

The check is **step-aware**: a parquet is up-to-date only if it contains rows tagged with the current `STEP`. Changing `STEP` and re-running overwrites mismatched files (`generate_angles` does not append). The largest N per source doubles as the ethalon for the convergence plot. For san at 100k images this is the slowest step; originals reuse existing `prep_cache_*_n360.npy` caches so N=360 is fast.

In [ ]:
GEN_KW = dict(workers=20, angles_tol=3, min_segment_len=5.0,
              keep_contours=False, chunksize=64)

for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        data.generate_angles_sweep(
            h5_path, sweep_dir(h5_path),
            ns=N_VALUES[kind][resolution], step=STEP,
            types_dict=types_dict, tag=kind,
            run_meta={'family': kind, 'resolution': resolution, 'tags': [], 'notes': ''},
            **GEN_KW,
        )

# Convergence grid

Walk every (resolution, kind) sweep folder once, compare each sample-size parquet against its resolution's ethalon (largest-N real parquet), and collect:

- `records_by_panel[(resolution, kind)]` — feeds the 3×3 plot below (rows = resolution, cols = grain class).
- `df_metrics` — long-form table for the statistical tests in the next section.

The 1024 row currently has only the original baseline (no san/diffit h5s at 1024). Real self-vs-ethalon at N_max is exactly zero by construction; it's dropped so the log-y axis stays anchored.

# Statistical tests: do metrics improve with N?

Per (kind, resolution, class) curve, test whether the metric magnitude decreases with N using a one-sided Kendall τ on `(N, |metric|)` (`alternative='less'`). For `w_dist` (already non-negative) we use the raw value; for signed Gaussian-fit relatives (`mu1/mu2/sigma1/sigma2/amp1/amp2`) we use absolute value, since closer to 0 = closer to the ethalon.

**`rel_err_abs_%`** = `(|m|@N_lo − |m|@N_hi) / |m|@N_lo × 100` — positive = |m| shrank between endpoints. Endpoints are kind-specific:

- **real**: sweep is `[100, 150, 200, 250, 300, 360]` — 6 points kept; the N=360 self-vs-ethalon zero anchors the plateau fit. Endpoint range stays **N_lo=100, N_hi=300** (using N=360 as N_hi would make `rel_err_%` and `alpha` degenerate).
- **san / diffit**: 8 points, **N_lo=1000, N_hi=10000** (the regime where convergence flattens).

Aggregations combine one-sided p-values via Fisher's method: per (kind, resolution), per (kind, metric), per metric, per kind. Each row carries `mean_rel_err_abs_%` so significance and effect size sit side by side.

In [ ]:
def collect_records():
    """Walk SOURCES once and return (records_by_panel, df_metrics).

    Per panel ethalon is the resolution's 360-image real parquet, so every
    san/diffit curve is (N sampled) vs (360 real). The 'real' kind ends up
    as a self-convergence baseline (N real vs 360 real).
    """
    records_by_panel, rows = {}, []
    for resolution, group in SOURCES.items():
        ethalon = find_ethalon(sweep_dir(group['real']))
        if ethalon is None:
            print(f'[row {resolution}] no ethalon parquet — skip row')
            continue
        print(f'row {resolution}: ethalon = {ethalon}')
        for kind, h5 in group.items():
            d = sweep_dir(h5)
            if not d.exists():
                continue
            recs = metrics_vs_n(d, str(ethalon),
                                class_map=CLASS_MAP_PER_KIND.get(kind, {}),
                                step=STEP,
                                allowed_ns=set(N_VALUES[kind][resolution]))
            records_by_panel[(resolution, kind)] = recs
            rows.extend({'kind': kind, 'resolution': resolution, **r} for r in recs)
    return records_by_panel, pd.DataFrame.from_records(rows)

records_by_panel, df_metrics = collect_records()

result = convergence_stats(df_metrics, METRICS, ENDPOINTS_BY_KIND,
                           expected_points={'real': 6, 'san': 8, 'diffit': 8},
                           pre_endpoints_by_kind=PRE_ENDPOINTS_BY_KIND)

# Per-subplot annotation: per-kind Kendall p, plateau-fit a/b for
# |m|(N) = a + b/sqrt(N), and its R^2. The plot fn colors each kind's label
# to match its curve, so the box doubles as a per-panel legend.
_STAT_COLS = ['kendall_p', 'a_hat', 'b_hat', 'fit_r2']

def _panel_stats(sub):
    return {row['kind']: {col: row[col] for col in _STAT_COLS}
            for _, row in sub.iterrows()}

# y-axis label per metric. w_dist is non-negative magnitude; the rest are
# signed relative errors and get a y=0 reference line in each panel.
METRIC_LABEL = {
    'w_dist': 'W-dist',
    'mu1':    'μ₁ rel. err.',
    'mu2':    'μ₂ rel. err.',
    'sigma1': 'σ₁ rel. err.',
    'sigma2': 'σ₂ rel. err.',
    'amp1':   'A₁ rel. err.',
    'amp2':   'A₂ rel. err.',
}

for metric in METRICS:
    sub_metric = result[result['metric'] == metric]
    panel_ann = {(int(res), cls): _panel_stats(g)
                 for (res, cls), g in sub_metric.groupby(['resolution', 'class'])}
    save_path = f'{metric}_convergence_step{STEP}.png'
    fig = plot_wdist_convergence_grid(
        records_by_panel, classes=CLASSES,
        kind_labels=LABELS, grain_labels=GRAIN_LABEL,
        row_keys=list(SOURCES), save_path=save_path,
        metric=metric, y_label=METRIC_LABEL[metric],
        zero_line=(metric != 'w_dist'),
        panel_annotations=panel_ann,
        annot_kind_labels=KIND_DISPLAY,
    )
    print(f'saved: {save_path}')
    fig.show()

print_convergence_report(result, METRICS, KINDS, ENDPOINTS_BY_KIND,
                         step=STEP, kind_display=KIND_DISPLAY,
                         pre_endpoints_by_kind=PRE_ENDPOINTS_BY_KIND)

# Gain distribution

Per-kind distribution of `gain_pct` (sampling-only gain remaining from N_max to the plateau, in %), colored by resolution. Companion to the per-curve report above: shows the generator-wide spread of how much room is left for sampling alone to close the gap to the bias floor.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from combra.stats.preprocess import stats_preprocess

STEP_PCT   = 1.5      # bin width for gain_% subplots (percentage points)
STEP_ALPHA = 0.1      # bin width for alpha subplots (dimensionless)
Y_LIM         = (0.0, 0.5)
GAIN_X_LIM    = (-20, 60)
ALPHA_X_LIM   = (-1.0, 2.0)

RESOLUTIONS = [256, 512]
# Column names match T3 of the convergence report (gain_%_a->b / alpha_a->b).
GAIN_COLS  = [('pre_rel_err_abs_%', 'gain_%_360->10^3'),
              ('rel_err_abs_%',     'gain_%_10^3->10^4')]
ALPHA_COLS = [('pre_alpha', 'alpha_360->10^3'),
              ('alpha',     'alpha_10^3->10^4')]
KIND_COLOR = {'san':    'rgba(31,119,180,0.95)',
              'diffit': 'rgba(255,127,14,0.95)'}

for col, label in GAIN_COLS + ALPHA_COLS:
    print(f'\n  [{label}]')
    for tag, line in summarize_metric_distribution(result, col, KINDS, RESOLUTIONS).items():
        print(f'    {tag}:  {line}')

_AXIS_KW = dict(showgrid=True, gridcolor='#DDDDDD', gridwidth=1,
                showline=True, linecolor='black', linewidth=1.2, mirror=True)

def _dist_figure(cols, bin_step, x_lim, ref_lines):
    # 2x2 distribution grid: rows=RESOLUTIONS, cols=`cols`, overlay by kind.
    fig = make_subplots(
        rows=len(RESOLUTIONS), cols=len(cols),
        shared_xaxes=True, shared_yaxes=True,
        subplot_titles=[f'{res}x{res} — {label}'
                        for res in RESOLUTIONS for _, label in cols],
        horizontal_spacing=0.08, vertical_spacing=0.14,
    )
    shown = set()
    for r, res in enumerate(RESOLUTIONS, start=1):
        for c, (col, _) in enumerate(cols, start=1):
            for kind in KINDS:
                vals = (result[(result['kind'] == kind) & (result['resolution'] == res)]
                        [col].dropna().values)
                if len(vals) == 0:
                    continue
                x, y = stats_preprocess(vals, bin_step)
                fig.add_trace(
                    go.Scatter(
                        x=x, y=y, mode='lines+markers',
                        line=dict(color=KIND_COLOR[kind], dash='dash', width=2),
                        marker=dict(color=KIND_COLOR[kind], size=12,
                                    line=dict(color='black', width=0.8)),
                        name=KIND_DISPLAY[kind], legendgroup=kind,
                        showlegend=(kind not in shown),
                    ), row=r, col=c)
                shown.add(kind)
            for x_ref, dash in ref_lines:
                fig.add_vline(x=x_ref, line=dict(color='gray', dash=dash, width=1),
                              row=r, col=c)
    for r in range(1, len(RESOLUTIONS) + 1):
        for c in range(1, len(cols) + 1):
            fig.update_xaxes(range=list(x_lim), zeroline=False, row=r, col=c, **_AXIS_KW)
            fig.update_yaxes(range=list(Y_LIM), zeroline=False, row=r, col=c, **_AXIS_KW)
        fig.update_yaxes(title_text='p(x)', row=r, col=1)
    for c, (_, label) in enumerate(cols, start=1):
        fig.update_xaxes(title_text=label, row=len(RESOLUTIONS), col=c)
    fig.update_layout(
        height=800, width=1200,
        plot_bgcolor='white', paper_bgcolor='white',
        legend=dict(bgcolor='rgba(255,255,255,0.92)',
                    bordercolor='black', borderwidth=1),
        margin=dict(l=80, r=30, t=70, b=70),
    )
    return fig

fig_gain = _dist_figure(GAIN_COLS, STEP_PCT, GAIN_X_LIM,
                        ref_lines=[(0.0, 'dash')])
save_gain = f'gain_dist_step{STEP}_binstep{STEP_PCT}.png'
fig_gain.write_image(save_gain, scale=2)
print(f'saved: {save_gain}')
fig_gain.show()

# alpha refs: 0 = no improvement, 0.5 = ideal Monte-Carlo decay.
fig_alpha = _dist_figure(ALPHA_COLS, STEP_ALPHA, ALPHA_X_LIM,
                         ref_lines=[(0.0, 'dash'), (0.5, 'dot')])
save_alpha = f'alpha_dist_step{STEP}_binstep{STEP_ALPHA}.png'
fig_alpha.write_image(save_alpha, scale=2)
print(f'saved: {save_alpha}')
fig_alpha.show()